# Entrenar el checkpoint NLLB-200 + LoRA (F. Prado) para retrotraducción

Este checkpoint es lo que necesita `4_aumento_datos/retrotraduccion.py` para traducir
las paráfrasis en español al shiwilu (paso 2 de la técnica).

Antes de correr: `Entorno de ejecución` -> `Cambiar tipo de entorno de ejecución` -> **GPU**.

## 1. Montar Google Drive (para guardar el checkpoint antes de que se cierre la sesión)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clonar tu repo y el de F. Prado

In [ ]:
GITHUB_USUARIO = "TU-USUARIO-AQUI"  # <-- cambia esto

!git clone https://github.com/{GITHUB_USUARIO}/shiwilu-tesis.git
%cd shiwilu-tesis
!git clone https://github.com/fapi19/Tesis_Spa-Jeb.git 4_aumento_datos/tesis_spa_jeb

## 3. Instalar dependencias (las que F. Prado ya fijó y probó)

In [ ]:
%cd /content/shiwilu-tesis/4_aumento_datos/tesis_spa_jeb
!pip install -q -r requirements/nmt.txt

## 4. Entrenar la configuración campeona (v2.1b LoRA+)

Esto puede tardar 30 min - 2 horas segun la GPU que te toque.

In [ ]:
!python -m scripts.nmt.30_train_lora \
    --variant xl \
    --rank 32 \
    --alpha 64 \
    --loraplus-lr-ratio 16 \
    --output-dir models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl

## 5. Guardar el checkpoint en Drive (¡no te saltes este paso!)

In [ ]:
!mkdir -p /content/drive/MyDrive/shiwilu_checkpoint
!cp -r models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl /content/drive/MyDrive/shiwilu_checkpoint/

## 6. Evaluar (deberia dar chrF++ promedio cercano a 44.99)

In [ ]:
!python -m scripts.nmt.40_evaluate --checkpoint models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl --split test

## 7. Correr la retrotraducción con tu script

Ya con el checkpoint entrenado, corre la técnica completa (parafraseo + traducción a shiwilu + filtros).

In [ ]:
%cd /content/shiwilu-tesis
!pip install -q sentence-transformers
!python 4_aumento_datos/retrotraduccion.py \
    --checkpoint 4_aumento_datos/tesis_spa_jeb/models/nmt/nllb_bidi_lora_v2_1b_loraplus_xl

## 8. Descargar el resultado a tu computadora

Descarga solo el CSV generado (no el checkpoint completo, pesa varios GB).

In [ ]:
from google.colab import files
files.download('4_aumento_datos/salidas/retrotraduccion.csv')